In [2]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# This notebook is inside RandomForest/, so use "."
ROOT = Path(".")

PKL_DIRS = [
    ROOT / "TCGA No Infinium" / "PKL",
    ROOT / "TCGA with InfiniumPurify" / "PKL",
    ROOT / "Norway No Infinium" / "PKL",
    ROOT / "Norway with InfiniumPurify" / "PKL",
]

pkl_paths = []
for pkl_dir in PKL_DIRS:
    if pkl_dir.exists():
        pkl_paths.extend(sorted(pkl_dir.glob("*.pkl")))

def to_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, tuple):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, list):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}

    if hasattr(obj, "get_params"):
        return repr(obj)

    return str(obj)

def shape_or_none(x):
    return tuple(x.shape) if hasattr(x, "shape") else None

def find_step(model, step_name):
    if hasattr(model, "named_steps") and step_name in model.named_steps:
        return model.named_steps[step_name]
    return None

def get_dataset_info(rel_path):
    top_folder = rel_path.parts[0]

    cohort = "TCGA" if "TCGA" in top_folder else "Norway"
    infinium_purify = "with InfiniumPurify" in top_folder

    return {
        "group_folder": top_folder,
        "cohort": cohort,
        "infinium_purify": infinium_purify,
    }

summary_rows = []
all_params = {}

for full_path in pkl_paths:
    rel_path = full_path.relative_to(ROOT)
    model_name = full_path.stem
    info = get_dataset_info(rel_path)

    model = joblib.load(full_path)
    file_size_bytes = full_path.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)

    has_named_steps = hasattr(model, "named_steps")
    step_names = list(model.named_steps.keys()) if has_named_steps else []

    irus_step = find_step(model, "irus")
    smote_step = find_step(model, "smote")
    rfe_step = find_step(model, "rfe")
    rf_step = find_step(model, "rf") if has_named_steps else model

    sampler_step = irus_step if irus_step is not None else smote_step
    sampler_name = "irus" if irus_step is not None else ("smote" if smote_step is not None else None)

    estimators = getattr(rf_step, "estimators_", None)
    feature_importances = getattr(rf_step, "feature_importances_", None)
    classes = getattr(rf_step, "classes_", None)
    n_features_in = getattr(rf_step, "n_features_in_", None)

    if estimators is not None and len(estimators) > 0:
        node_counts = [est.tree_.node_count for est in estimators]
        max_depths = [est.tree_.max_depth for est in estimators]
        n_leaves = [est.tree_.n_leaves for est in estimators]

        total_nodes = int(np.sum(node_counts))
        avg_nodes_per_tree = float(np.mean(node_counts))
        avg_tree_depth = float(np.mean(max_depths))
        max_tree_depth = int(np.max(max_depths))
        avg_leaves_per_tree = float(np.mean(n_leaves))
    else:
        total_nodes = None
        avg_nodes_per_tree = None
        avg_tree_depth = None
        max_tree_depth = None
        avg_leaves_per_tree = None

    summary_rows.append({
        "model_name": model_name,
        "relative_path": str(rel_path),
        "group_folder": info["group_folder"],
        "cohort": info["cohort"],
        "infinium_purify": info["infinium_purify"],

        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),

        "object_type": type(model).__name__,
        "steps": " -> ".join(step_names) if step_names else None,

        "sampler_name": sampler_name,
        "sampler_type": type(sampler_step).__name__ if sampler_step is not None else None,
        "sampling_strategy": str(getattr(sampler_step, "sampling_strategy", None)) if sampler_step is not None else None,

        "rfe_present": rfe_step is not None,
        "rfe_n_features_selected": getattr(rfe_step, "n_features_", None) if rfe_step is not None else None,
        "rfe_support_count": int(np.sum(rfe_step.support_)) if rfe_step is not None and hasattr(rfe_step, "support_") else None,
        "rfe_step_size": getattr(rfe_step, "step", None) if rfe_step is not None else None,

        "rf_type": type(rf_step).__name__,
        "n_estimators": getattr(rf_step, "n_estimators", None),
        "criterion": getattr(rf_step, "criterion", None),
        "max_depth": getattr(rf_step, "max_depth", None),
        "max_features": str(getattr(rf_step, "max_features", None)),
        "min_samples_split": getattr(rf_step, "min_samples_split", None),
        "min_samples_leaf": getattr(rf_step, "min_samples_leaf", None),
        "bootstrap": getattr(rf_step, "bootstrap", None),
        "class_weight": str(getattr(rf_step, "class_weight", None)),
        "random_state": getattr(rf_step, "random_state", None),

        "n_features_in_": n_features_in,
        "n_classes": len(classes) if classes is not None else None,
        "classes": str(classes.tolist() if hasattr(classes, "tolist") else classes),
        "feature_importances_len": len(feature_importances) if feature_importances is not None else None,
        "feature_importances_sum": float(np.sum(feature_importances)) if feature_importances is not None else None,

        "trained_tree_count": len(estimators) if estimators is not None else None,
        "total_nodes": total_nodes,
        "avg_nodes_per_tree": round(avg_nodes_per_tree, 2) if avg_nodes_per_tree is not None else None,
        "avg_tree_depth": round(avg_tree_depth, 2) if avg_tree_depth is not None else None,
        "max_tree_depth": max_tree_depth,
        "avg_leaves_per_tree": round(avg_leaves_per_tree, 2) if avg_leaves_per_tree is not None else None,
    })

    all_params[model_name] = {
        "relative_path": str(rel_path),
        "group_folder": info["group_folder"],
        "cohort": info["cohort"],
        "infinium_purify": info["infinium_purify"],

        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),

        "model_type": type(model).__name__,
        "pipeline_steps": step_names,
        "pipeline_params": to_serializable(model.get_params(deep=True)) if hasattr(model, "get_params") else None,

        "sampler_name": sampler_name,
        "sampler_params": to_serializable(sampler_step.get_params(deep=True)) if sampler_step is not None and hasattr(sampler_step, "get_params") else None,

        "rfe_params": to_serializable(rfe_step.get_params(deep=True)) if rfe_step is not None and hasattr(rfe_step, "get_params") else None,
        "rf_params": to_serializable(rf_step.get_params(deep=True)) if hasattr(rf_step, "get_params") else None,

        "trained_attributes": {
            "classes_": to_serializable(classes),
            "n_features_in_": to_serializable(n_features_in),
            "feature_importances_shape": to_serializable(shape_or_none(feature_importances)),
            "feature_importances_sum": to_serializable(float(np.sum(feature_importances)) if feature_importances is not None else None),
            "trained_tree_count": to_serializable(len(estimators) if estimators is not None else None),
            "total_nodes": to_serializable(total_nodes),
            "avg_nodes_per_tree": to_serializable(avg_nodes_per_tree),
            "avg_tree_depth": to_serializable(avg_tree_depth),
            "max_tree_depth": to_serializable(max_tree_depth),
            "avg_leaves_per_tree": to_serializable(avg_leaves_per_tree),
        }
    }

    if rfe_step is not None:
        all_params[model_name]["rfe_trained_attributes"] = {
            "n_features_": to_serializable(getattr(rfe_step, "n_features_", None)),
            "support_sum": to_serializable(int(np.sum(rfe_step.support_)) if hasattr(rfe_step, "support_") else None),
            "ranking_shape": to_serializable(shape_or_none(getattr(rfe_step, "ranking_", None))),
        }

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["cohort", "infinium_purify", "relative_path"]
).reset_index(drop=True)

pd.set_option("display.max_columns", None)
display(summary_df)

summary_csv = ROOT / "rf_model_summary.csv"
params_json = ROOT / "rf_model_params.json"

summary_df.to_csv(summary_csv, index=False)

with open(params_json, "w", encoding="utf-8") as f:
    json.dump(all_params, f, indent=2, ensure_ascii=False)

print("\nSaved:")
print(summary_csv)
print(params_json)

print("\nExample:")
first_key = next(iter(all_params))
print("Model:", first_key)
print(json.dumps(all_params[first_key], indent=2, ensure_ascii=False))


c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. Thi

,model_name,relative_path,group_folder,cohort,infinium_purify,file_size_bytes,file_size_mb,object_type,steps,sampler_name,sampler_type,sampling_strategy,rfe_present,rfe_n_features_selected,rfe_support_count,rfe_step_size,rf_type,n_estimators,criterion,max_depth,max_features,min_samples_split,min_samples_leaf,bootstrap,class_weight,random_state,n_features_in_,n_classes,classes,feature_importances_len,feature_importances_sum,trained_tree_count,total_nodes,avg_nodes_per_tree,avg_tree_depth,max_tree_depth,avg_leaves_per_tree
0,RF_Norway_IRUS_No_RFE,Norway No Infinium\PKL\RF_Norway_IRUS_No_RFE.pkl,Norway No Infinium,Norway,False,1312150,1.25,Pipeline,irus -> rf,irus,RandomUnderSampler,{'LumA': 30},False,NaN,NaN,NaN,RandomForestClassifier,181,gini,20.0,sqrt,7,1,True,None,0,20789,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",20789,1.0,181,5549,30.66,6.57,10,15.83
1,RF_Norway_IRUS_RFE,Norway No Infinium\PKL\RF_Norway_IRUS_RFE.pkl,Norway No Infinium,Norway,False,2072390,1.98,Pipeline,irus -> rfe -> rf,irus,RandomUnderSampler,{'LumA': 30},True,100.0,100.0,0.50,RandomForestClassifier,145,gini,NaN,sqrt,5,1,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,145,5657,39.01,7.52,12,20.01
2,RF_Norway_NoImb_No_RFE,Norway No Infinium\PKL\RF_Norway_NoImb_No_RFE.pkl,Norway No Infinium,Norway,False,917174,0.87,Pipeline,rf,None,None,None,False,NaN,NaN,NaN,RandomForestClassifier,112,gini,15.0,sqrt,4,2,True,None,0,20789,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",20789,1.0,112,5196,46.39,8.18,11,23.70
3,RF_Norway_NoImb_RFE,Norway No Infinium\PKL\RF_Norway_NoImb_RFE.pkl,Norway No Infinium,Norway,False,2103718,2.01,Pipeline,rfe -> rf,None,None,None,True,100.0,100.0,0.50,RandomForestClassifier,153,gini,20.0,sqrt,5,2,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,153,7439,48.62,8.83,12,24.81
4,RF_Norway_SMOTE_No_RFE,Norway No Infinium\PKL\RF_Norway_SMOTE_No_RFE.pkl,Norway No Infinium,Norway,False,4143910,3.95,Pipeline,smote -> rf,smote,SMOTE,auto,False,NaN,NaN,NaN,RandomForestClassifier,245,gini,25.0,sqrt,6,1,True,None,0,20789,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",20789,1.0,245,16533,67.48,8.93,13,34.24
5,RF_Norway_SMOTE_RFE,Norway No Infinium\PKL\RF_Norway_SMOTE_RFE.pkl,Norway No Infinium,Norway,False,5053238,4.82,Pipeline,smote -> rfe -> rf,smote,SMOTE,auto,True,500.0,500.0,0.50,RandomForestClassifier,227,gini,NaN,sqrt,2,4,True,None,0,500,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",500,1.0,227,13883,61.16,8.80,13,31.08
6,RF_Norway_IP_IRUS_No_RFE,Norway with InfiniumPurify\PKL\RF_Norway_IP_IR...,Norway with InfiniumPurify,Norway,True,915702,0.87,Pipeline,irus -> rf,irus,RandomUnderSampler,{'LumA': 36},False,NaN,NaN,NaN,RandomForestClassifier,63,gini,15.0,sqrt,5,3,True,None,0,21101,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",21101,1.0,63,2119,33.63,6.63,9,17.32
7,RF_Norway_IP_IRUS_RFE,Norway with InfiniumPurify\PKL\RF_Norway_IP_IR...,Norway with InfiniumPurify,Norway,True,2665606,2.54,Pipeline,irus -> rfe -> rf,irus,RandomUnderSampler,{'LumA': 36},True,100.0,100.0,0.25,RandomForestClassifier,258,gini,25.0,log2,3,3,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,258,10158,39.37,7.27,10,20.19
8,RF_Norway_IP_NoImb_No_RFE,Norway with InfiniumPurify\PKL\RF_Norway_IP_No...,Norway with InfiniumPurify,Norway,True,968950,0.92,Pipeline,rf,None,None,None,False,NaN,NaN,NaN,RandomForestClassifier,146,gini,15.0,sqrt,8,2,True,None,0,21101,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",21101,1.0,146,5506,37.71,8.21,12,19.36
9,RF_Norway_IP_NoImb_RFE,Norway with InfiniumPurify\PKL\RF_Norway_IP_No...,Norway with InfiniumPurify,Norway,True,2713718,2.59,Pipeline,rfe -> rf,None,None,None,True,100.0,100.0,0.25,RandomForestClassifier,258,gini,25.0,log2,3,3,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,258,12550,48.64,8.70,13,24.82



Saved:
rf_model_summary.csv
rf_model_params.json

Example:
Model: RF_TCGA_IRUS_No_RFE
{
  "relative_path": "TCGA No Infinium\\PKL\\RF_TCGA_IRUS_No_RFE.pkl",
  "group_folder": "TCGA No Infinium",
  "cohort": "TCGA",
  "infinium_purify": false,
  "file_size_bytes": 1646998,
  "file_size_mb": 1.57,
  "model_type": "Pipeline",
  "pipeline_steps": [
    "irus",
    "rf"
  ],
  "pipeline_params": {
    "memory": null,
    "steps": [
      [
        "irus",
        "RandomUnderSampler(random_state=0, sampling_strategy={'LumA': 74})"
      ],
      [
        "rf",
        "RandomForestClassifier(max_depth=20, min_samples_split=7, n_estimators=181,\n                       random_state=0)"
      ]
    ],
    "transform_input": null,
    "verbose": false,
    "irus": "RandomUnderSampler(random_state=0, sampling_strategy={'LumA': 74})",
    "rf": "RandomForestClassifier(max_depth=20, min_samples_split=7, n_estimators=181,\n                       random_state=0)",
    "irus__random_state": 0,
    

In [4]:
from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd

# This notebook is inside RandomForest/, so use "."
ROOT = Path(".")

# Only the specific RF models you asked for
SELECTED_MODELS = {
    "tcga_smote_rfe": ROOT / "TCGA No Infinium" / "PKL" / "RF_TCGA_SMOTE_RFE.pkl",
    "tcga_purified_smote_rfe": ROOT / "TCGA with InfiniumPurify" / "PKL" / "RF_TCGA_IP_SMOTE_RFE.pkl",
    "norway_noimb": ROOT / "Norway No Infinium" / "PKL" / "RF_Norway_NoImb_No_RFE.pkl",
    "norway_purified_irus_rfe": ROOT / "Norway with InfiniumPurify" / "PKL" / "RF_Norway_IP_IRUS_RFE.pkl",
}

pkl_paths = []
missing_paths = []

for label, path in SELECTED_MODELS.items():
    if path.exists():
        pkl_paths.append((label, path))
    else:
        missing_paths.append((label, str(path)))

def to_serializable(obj):
    if obj is None:
        return None

    if isinstance(obj, (str, int, float, bool)):
        return obj

    if isinstance(obj, Path):
        return str(obj)

    if isinstance(obj, (np.integer,)):
        return int(obj)

    if isinstance(obj, (np.floating,)):
        return float(obj)

    if isinstance(obj, (np.bool_,)):
        return bool(obj)

    if isinstance(obj, np.ndarray):
        return obj.tolist()

    if isinstance(obj, tuple):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, list):
        return [to_serializable(x) for x in obj]

    if isinstance(obj, dict):
        return {str(k): to_serializable(v) for k, v in obj.items()}

    if hasattr(obj, "get_params"):
        return repr(obj)

    return str(obj)

def shape_or_none(x):
    return tuple(x.shape) if hasattr(x, "shape") else None

def find_step(model, step_name):
    if hasattr(model, "named_steps") and step_name in model.named_steps:
        return model.named_steps[step_name]
    return None

def get_dataset_info(rel_path):
    top_folder = rel_path.parts[0]

    cohort = "TCGA" if "TCGA" in top_folder else "Norway"
    infinium_purify = "with InfiniumPurify" in top_folder

    return {
        "group_folder": top_folder,
        "cohort": cohort,
        "infinium_purify": infinium_purify,
    }

summary_rows = []
all_params = {}

for selected_label, full_path in pkl_paths:
    rel_path = full_path.relative_to(ROOT)
    model_name = full_path.stem
    info = get_dataset_info(rel_path)

    model = joblib.load(full_path)
    file_size_bytes = full_path.stat().st_size
    file_size_mb = file_size_bytes / (1024 ** 2)

    has_named_steps = hasattr(model, "named_steps")
    step_names = list(model.named_steps.keys()) if has_named_steps else []

    irus_step = find_step(model, "irus")
    smote_step = find_step(model, "smote")
    rfe_step = find_step(model, "rfe")
    rf_step = find_step(model, "rf") if has_named_steps else model

    sampler_step = irus_step if irus_step is not None else smote_step
    sampler_name = "irus" if irus_step is not None else ("smote" if smote_step is not None else None)

    estimators = getattr(rf_step, "estimators_", None)
    feature_importances = getattr(rf_step, "feature_importances_", None)
    classes = getattr(rf_step, "classes_", None)
    n_features_in = getattr(rf_step, "n_features_in_", None)

    if estimators is not None and len(estimators) > 0:
        node_counts = [est.tree_.node_count for est in estimators]
        max_depths = [est.tree_.max_depth for est in estimators]
        n_leaves = [est.tree_.n_leaves for est in estimators]

        total_nodes = int(np.sum(node_counts))
        avg_nodes_per_tree = float(np.mean(node_counts))
        avg_tree_depth = float(np.mean(max_depths))
        max_tree_depth = int(np.max(max_depths))
        avg_leaves_per_tree = float(np.mean(n_leaves))
    else:
        total_nodes = None
        avg_nodes_per_tree = None
        avg_tree_depth = None
        max_tree_depth = None
        avg_leaves_per_tree = None

    summary_rows.append({
        "selected_label": selected_label,
        "model_name": model_name,
        "relative_path": str(rel_path),
        "group_folder": info["group_folder"],
        "cohort": info["cohort"],
        "infinium_purify": info["infinium_purify"],

        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),

        "object_type": type(model).__name__,
        "steps": " -> ".join(step_names) if step_names else None,

        "sampler_name": sampler_name,
        "sampler_type": type(sampler_step).__name__ if sampler_step is not None else None,
        "sampling_strategy": str(getattr(sampler_step, "sampling_strategy", None)) if sampler_step is not None else None,

        "rfe_present": rfe_step is not None,
        "rfe_n_features_selected": getattr(rfe_step, "n_features_", None) if rfe_step is not None else None,
        "rfe_support_count": int(np.sum(rfe_step.support_)) if rfe_step is not None and hasattr(rfe_step, "support_") else None,
        "rfe_step_size": getattr(rfe_step, "step", None) if rfe_step is not None else None,

        "rf_type": type(rf_step).__name__,
        "n_estimators": getattr(rf_step, "n_estimators", None),
        "criterion": getattr(rf_step, "criterion", None),
        "max_depth": getattr(rf_step, "max_depth", None),
        "max_features": str(getattr(rf_step, "max_features", None)),
        "min_samples_split": getattr(rf_step, "min_samples_split", None),
        "min_samples_leaf": getattr(rf_step, "min_samples_leaf", None),
        "bootstrap": getattr(rf_step, "bootstrap", None),
        "class_weight": str(getattr(rf_step, "class_weight", None)),
        "random_state": getattr(rf_step, "random_state", None),

        "n_features_in_": n_features_in,
        "n_classes": len(classes) if classes is not None else None,
        "classes": str(classes.tolist() if hasattr(classes, "tolist") else classes),
        "feature_importances_len": len(feature_importances) if feature_importances is not None else None,
        "feature_importances_sum": float(np.sum(feature_importances)) if feature_importances is not None else None,

        "trained_tree_count": len(estimators) if estimators is not None else None,
        "total_nodes": total_nodes,
        "avg_nodes_per_tree": round(avg_nodes_per_tree, 2) if avg_nodes_per_tree is not None else None,
        "avg_tree_depth": round(avg_tree_depth, 2) if avg_tree_depth is not None else None,
        "max_tree_depth": max_tree_depth,
        "avg_leaves_per_tree": round(avg_leaves_per_tree, 2) if avg_leaves_per_tree is not None else None,
    })

    all_params[selected_label] = {
        "model_name": model_name,
        "relative_path": str(rel_path),
        "group_folder": info["group_folder"],
        "cohort": info["cohort"],
        "infinium_purify": info["infinium_purify"],

        "file_size_bytes": int(file_size_bytes),
        "file_size_mb": round(file_size_mb, 2),

        "model_type": type(model).__name__,
        "pipeline_steps": step_names,
        "pipeline_params": to_serializable(model.get_params(deep=True)) if hasattr(model, "get_params") else None,

        "sampler_name": sampler_name,
        "sampler_params": to_serializable(sampler_step.get_params(deep=True)) if sampler_step is not None and hasattr(sampler_step, "get_params") else None,

        "rfe_params": to_serializable(rfe_step.get_params(deep=True)) if rfe_step is not None and hasattr(rfe_step, "get_params") else None,
        "rf_params": to_serializable(rf_step.get_params(deep=True)) if hasattr(rf_step, "get_params") else None,

        "trained_attributes": {
            "classes_": to_serializable(classes),
            "n_features_in_": to_serializable(n_features_in),
            "feature_importances_shape": to_serializable(shape_or_none(feature_importances)),
            "feature_importances_sum": to_serializable(float(np.sum(feature_importances)) if feature_importances is not None else None),
            "trained_tree_count": to_serializable(len(estimators) if estimators is not None else None),
            "total_nodes": to_serializable(total_nodes),
            "avg_nodes_per_tree": to_serializable(avg_nodes_per_tree),
            "avg_tree_depth": to_serializable(avg_tree_depth),
            "max_tree_depth": to_serializable(max_tree_depth),
            "avg_leaves_per_tree": to_serializable(avg_leaves_per_tree),
        }
    }

    if rfe_step is not None:
        all_params[selected_label]["rfe_trained_attributes"] = {
            "n_features_": to_serializable(getattr(rfe_step, "n_features_", None)),
            "support_sum": to_serializable(int(np.sum(rfe_step.support_)) if hasattr(rfe_step, "support_") else None),
            "ranking_shape": to_serializable(shape_or_none(getattr(rfe_step, "ranking_", None))),
        }

summary_df = pd.DataFrame(summary_rows).sort_values(
    ["cohort", "infinium_purify", "relative_path"]
).reset_index(drop=True)

pd.set_option("display.max_columns", None)

if missing_paths:
    print("Missing selected files:")
    for label, path in missing_paths:
        print(f"  {label}: {path}")

display(summary_df)

summary_csv = ROOT / "rf_model_summary_selected.csv"
params_json = ROOT / "rf_model_params_selected.json"

summary_df.to_csv(summary_csv, index=False)

with open(params_json, "w", encoding="utf-8") as f:
    json.dump(all_params, f, indent=2, ensure_ascii=False)

print("\nSaved:")
print(summary_csv)
print(params_json)

print("\nExample:")
first_key = next(iter(all_params))
print("Selected label:", first_key)
print(json.dumps(all_params[first_key], indent=2, ensure_ascii=False))


c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator NearestNeighbors from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator DecisionTreeClassifier from version 1.6.1 when using version 1.8.0. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(
c:\Users\Public\Anaconda\Lib\site-packages\sklearn\base.py:463: InconsistentVersionWarning: Trying to unpickle estimator RandomForestClassifier from version 1.6.1 when using version 1.8.0. This migh

,selected_label,model_name,relative_path,group_folder,cohort,infinium_purify,file_size_bytes,file_size_mb,object_type,steps,sampler_name,sampler_type,sampling_strategy,rfe_present,rfe_n_features_selected,rfe_support_count,rfe_step_size,rf_type,n_estimators,criterion,max_depth,max_features,min_samples_split,min_samples_leaf,bootstrap,class_weight,random_state,n_features_in_,n_classes,classes,feature_importances_len,feature_importances_sum,trained_tree_count,total_nodes,avg_nodes_per_tree,avg_tree_depth,max_tree_depth,avg_leaves_per_tree
0,norway_noimb,RF_Norway_NoImb_No_RFE,Norway No Infinium\PKL\RF_Norway_NoImb_No_RFE.pkl,Norway No Infinium,Norway,False,917174,0.87,Pipeline,rf,None,None,None,False,NaN,NaN,NaN,RandomForestClassifier,112,gini,15.0,sqrt,4,2,True,None,0,20789,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",20789,1.0,112,5196,46.39,8.18,11,23.70
1,norway_purified_irus_rfe,RF_Norway_IP_IRUS_RFE,Norway with InfiniumPurify\PKL\RF_Norway_IP_IR...,Norway with InfiniumPurify,Norway,True,2665606,2.54,Pipeline,irus -> rfe -> rf,irus,RandomUnderSampler,{'LumA': 36},True,100.0,100.0,0.25,RandomForestClassifier,258,gini,25.0,log2,3,3,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,258,10158,39.37,7.27,10,20.19
2,tcga_smote_rfe,RF_TCGA_SMOTE_RFE,TCGA No Infinium\PKL\RF_TCGA_SMOTE_RFE.pkl,TCGA No Infinium,TCGA,False,16588726,15.82,Pipeline,smote -> rfe -> rf,smote,SMOTE,auto,True,100.0,100.0,0.20,RandomForestClassifier,218,gini,NaN,log2,5,1,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,218,31592,144.92,12.17,22,72.96
3,tcga_purified_smote_rfe,RF_TCGA_IP_SMOTE_RFE,TCGA with InfiniumPurify\PKL\RF_TCGA_IP_SMOTE_...,TCGA with InfiniumPurify,TCGA,True,15282006,14.57,Pipeline,smote -> rfe -> rf,smote,SMOTE,auto,True,100.0,100.0,0.20,RandomForestClassifier,181,gini,20.0,log2,6,4,True,None,0,100,5,"['Basal', 'Her2', 'LumA', 'LumB', 'Normal']",100,1.0,181,20759,114.69,11.01,15,57.85



Saved:
rf_model_summary_selected.csv
rf_model_params_selected.json

Example:
Selected label: tcga_smote_rfe
{
  "model_name": "RF_TCGA_SMOTE_RFE",
  "relative_path": "TCGA No Infinium\\PKL\\RF_TCGA_SMOTE_RFE.pkl",
  "group_folder": "TCGA No Infinium",
  "cohort": "TCGA",
  "infinium_purify": false,
  "file_size_bytes": 16588726,
  "file_size_mb": 15.82,
  "model_type": "Pipeline",
  "pipeline_steps": [
    "smote",
    "rfe",
    "rf"
  ],
  "pipeline_params": {
    "memory": null,
    "steps": [
      [
        "smote",
        "SMOTE(k_neighbors=3, random_state=0)"
      ],
      [
        "rfe",
        "RFE(estimator=RandomForestClassifier(random_state=0), n_features_to_select=100,\n    step=0.2)"
      ],
      [
        "rf",
        "RandomForestClassifier(max_features='log2', min_samples_split=5,\n                       n_estimators=218, random_state=0)"
      ]
    ],
    "transform_input": null,
    "verbose": false,
    "smote": "SMOTE(k_neighbors=3, random_state=0)",
    "